# BETO-only ablation — ROCCO committee revision (item 1)

Standalone companion to `Fake_news_detection_model_Final_Version.ipynb`.

**Goal:** does ROCCO actually beat the transformer it is built on — even a well-tuned BETO? This notebook replays the same data pipeline (fetch + augmentation) so the 8-fold `StratifiedKFold(random_state=42)` split is identical to the main notebook, then evaluates two text-only BETO configurations with **no graph**:

| Row | Test | What it shows |
|---|---|---|
| 1 | **BETO as-is** — frozen lower-6 layers, 3 epochs, predict from `cls_head` on test CLS embeddings | the bare transformer at ROCCO's exact text config |
| 2 | **Better-tuned BETO** — all layers trainable, 6 epochs | BETO's own ceiling, text-only |
| 3 | **Full ROCCO** *(reference)* | 0.924 acc / 0.926 macro-F1 |

> Note on reproducibility: augmented-row *labels* and fold membership are deterministic (seeded), so folds match across runs; the augmented *text* varies run-to-run because the masking uses Python's unseeded `random` (same behaviour as the original notebook). This run-to-run noise is already reflected in the reported ±std.

In [1]:
!pip install -q transformers

In [2]:
import gc
import numpy as np
import pandas as pd
import requests

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset

from transformers import AutoTokenizer, AutoModel

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


## Data loading & preprocessing (replicated from the main notebook)

These three cells are copied **verbatim** from `Fake_news_detection_model_Final_Version.ipynb` (fetch → label map → BETO fill-mask augmentation → tokenizer). Because the subset/shuffle are seeded (`random_state=42`) and augmented rows inherit their source label, the resulting `StratifiedKFold(random_state=42)` folds match the main notebook's exactly. The GAT metadata feature-engineering is intentionally omitted: the BETO baseline is text+label only and those features affect neither the labels nor the row order.

In [4]:
# API endpoint
ROUTE = "https://fake-news-data-extraction.onrender.com/dataset"

# Fetch the data
response = requests.get(ROUTE)
data = response.json()

# Extract 'Scrapped news' list
news_items = data["Scrapped news"]

# Convert to DataFrame
df = pd.DataFrame(news_items)

# Preprocess the dataset for model input
df['label'] = df['VERACIDAD'].map({'true': 0, 'false': 1, "satira":2})  # Map veracity to binary labels


df.drop(["METADATA", "VERACIDAD"], axis=1, inplace=True)

df.head()


,AUTOR,CORPUS,FECHA,TITULO,URL,label
0,Heraldo De Aragón,La Universidad de Oxford lanza en España su nu...,2017-04-05 00:00:00,Oxford lanza sus propios exámenes de certifica...,https://www.heraldo.es/noticias/sociedad/2017/...,0
1,Ernesto Agudo,"Darío Villanueva, director de la RAE, en una r...",2018-06-26 13:51:37+02:00,La RAE estudia incluir «machirulo» en el Dicci...,https://www.abc.es/cultura/abci-estudia-inclui...,0
2,"Pedro, Villa Y Caña, Pedro Villa Y Caña, Ver P...",Estudiantes de la F acultad de Ciencias Políti...,,Realizan paro en Facultad de Ciencias Política...,http://www.eluniversal.com.mx/nacion/sociedad/...,0
3,Dios De La Guerra Y Referente Del Humor Period...,This post has already been read 3168 times!\n\...,2018-01-18 08:47:27+00:00,Deniegan el B1 a un joven mudo por no poder ap...,https://haynoticia.es/deniegan-b1-joven-mudo-n...,1
4,"Redacción El Universal, Ver Perfil",Los granos de maíz que fueron tostados en el c...,,"mole, tradicional, familia, nahua, discriminad...",http://www.eluniversal.com.mx/orgullomexicano/...,0


In [5]:
from transformers import pipeline
import random, pandas as pd, requests

# Load BETO masked LM
fill_mask = pipeline("fill-mask", model="dccuchile/bert-base-spanish-wwm-cased", top_k=3)

def augment_text(text, mask_prob=0.15):
    """Replace ~15% of words with BETO predictions."""
    words = text.split()
    if len(words) < 5: return text
    n = max(1, int(len(words) * mask_prob))
    for i in random.sample(range(len(words)), n):
        words[i] = "[MASK]"
    masked = " ".join(words)
    try:
        preds = fill_mask(masked)
        if isinstance(preds, list):
            for p in preds:
                masked = masked.replace("[MASK]", p['token_str'], 1)
        return masked
    except:
        return text

# Apply augmentation to 30% of dataset (skip satire if desired)
subset = df[df["label"] != 2].sample(frac=0.3, random_state=42)
aug_df = subset.copy()
aug_df["CORPUS"] = aug_df["CORPUS"].apply(augment_text)

# Merge and shuffle
df = pd.concat([df, aug_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"✅ Data Augmentation complete: added {len(aug_df)} samples (total {len(df)})")
df.head()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (613 > 512). Running this sequence through the model will result in indexing errors
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✅ Data Augmentation complete: added 257 samples (total 1547)


,AUTOR,CORPUS,FECHA,TITULO,URL,label
0,"Efe, Foto","LONDRES.\n\nLa última teoría científica, sobre...",2018-05-02 14:47:59-06:00,Publican la última teoría de Stephen Hawking,https://www.excelsior.com.mx/global/publican-l...,0
1,"Aitor Tilla, Gabriela Serafino, Juan Pablo Seg...",La procrastinación está considerada como una d...,2013-11-19 12:16:54+00:00,La Procrastinación será ilegal,http://lavozpopular.com/procrastinacion-sera-i...,2
2,"Oso Rulo, Escrito Por",¿No puedes pasar cinco minutos sin hacer el ri...,2018-04-30 00:00:00,"¡Hazte a un lado, Ventaneando! Los de Hoy se h...",https://eldeforma.com/2018/04/30/hazte-un-lado...,2
3,Rocío Niebla,El embarazo es parte de nuestra sexualidad y v...,2021-02-12 00:00:00,Sexo durante el embarazo: esto es todo lo que ...,https://elpais.com/mamas-papas/2021-02-12/sexo...,0
4,"Gustavo Rugeles, Director De El Expediente",A comienzos de éste año todos los medios de co...,2019-10-02 15:39:08+00:00,"Video: redadas en el banco Vaticano, el Diacon...",https://elexpediente.co/video-redadas-el-banco...,1


In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    "dccuchile/bert-base-spanish-wwm-cased"
)

## BETO-only ablation (committee item 1)

Two BETO-only runs on the same 8-fold split, compared against the existing Full ROCCO number (0.924 acc / 0.926 macro-F1). Run on a GPU runtime and paste back the `ITEM 1 SUMMARY` block.

In [7]:
# =====================================================================
# BETO-only ablation (no GAT)
# Same 8-fold StratifiedKFold(random_state=42) as the ROCCO CV loop.
#   "BETO as-is"        -> frozen lower-6 layers, 3 epochs  (ROCCO's exact text config)
#   "Better-tuned BETO" -> all layers trainable, 6 epochs   (BETO's own ceiling)
#   Full ROCCO (reference, already have): 0.924 acc / 0.926 macro-F1
# Each config predicts straight from cls_head on the test CLS embeddings (no graph).
# =====================================================================
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

configs = [
    ("BETO as-is",        True,  3),   # (tag, freeze_lower6, epochs)
    ("Better-tuned BETO", False, 6),
]

y = df['label'].values
summary = {}

for tag, freeze_lower6, epochs in configs:
    print(f"\n##### {tag}  (freeze_lower6={freeze_lower6}, epochs={epochs}) #####")
    skf = StratifiedKFold(n_splits=8, shuffle=True, random_state=42)

    acc_scores, f1_scores = [], []
    all_trues, all_probs = [], []

    for fold, (train_idx, test_idx) in enumerate(skf.split(df, y)):
        df_tr = df.iloc[train_idx].copy()
        df_te = df.iloc[test_idx].copy()

        tr_enc = tokenizer(list(df_tr['CORPUS']), truncation=True, padding=True,
                           max_length=256, return_tensors="pt").to(device)
        te_enc = tokenizer(list(df_te['CORPUS']), truncation=True, padding=True,
                           max_length=256, return_tensors="pt").to(device)
        ytr = torch.tensor(df_tr['label'].values, dtype=torch.long).to(device)
        yte = torch.tensor(df_te['label'].values, dtype=torch.long).to(device)

        beto = AutoModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased").to(device)

        # Replicates the ROCCO loop's freezing expression verbatim when freeze_lower6=True.
        if freeze_lower6:
            for name, param in beto.named_parameters():
                if any(f"encoder.layer.{i}" in name for i in range(6)):
                    param.requires_grad = False

        cls_head = nn.Linear(768, 3).to(device)
        params = [p for p in beto.parameters() if p.requires_grad] + list(cls_head.parameters())
        opt = AdamW(params, lr=2e-5)
        crit = nn.CrossEntropyLoss(label_smoothing=0.1)
        train_loader = DataLoader(
            TensorDataset(tr_enc['input_ids'], tr_enc['attention_mask'], ytr),
            batch_size=8, shuffle=True)
        amp_scaler = torch.cuda.amp.GradScaler()

        # === TRAIN BETO ===
        beto.train(); cls_head.train()
        for epoch in range(epochs):
            for input_ids, attn, labels in train_loader:
                opt.zero_grad()
                with torch.cuda.amp.autocast():
                    out = beto(input_ids=input_ids, attention_mask=attn)
                    logits = cls_head(out.last_hidden_state[:, 0, :])
                    loss = crit(logits, labels)
                amp_scaler.scale(loss).backward()
                amp_scaler.step(opt)
                amp_scaler.update()

        # === BETO-only inference: cls_head on test CLS embeddings, no GAT ===
        beto.eval(); cls_head.eval()
        with torch.no_grad():
            with torch.cuda.amp.autocast():
                logits_te = cls_head(beto(**te_enc).last_hidden_state[:, 0, :]).float()
            probs = logits_te.softmax(dim=1).cpu().numpy()
            pred = logits_te.argmax(dim=1).cpu().numpy()
        true = yte.cpu().numpy()

        acc = accuracy_score(true, pred)
        f1 = f1_score(true, pred, average='macro')
        acc_scores.append(acc); f1_scores.append(f1)
        all_trues.extend(true); all_probs.append(probs)
        print(f"  Fold {fold + 1}: Acc={acc:.3f}  F1={f1:.3f}")

        del beto, cls_head, tr_enc, te_enc, train_loader
        torch.cuda.empty_cache(); gc.collect()

    all_probs = np.vstack(all_probs)
    y_bin = label_binarize(np.array(all_trues), classes=[0, 1, 2])
    roc_macro = roc_auc_score(y_bin, all_probs, average='macro', multi_class='ovr')
    summary[tag] = (np.mean(acc_scores), np.std(acc_scores),
                    np.mean(f1_scores), np.std(f1_scores), roc_macro)
    print(f"  -> Acc {np.mean(acc_scores):.4f}+/-{np.std(acc_scores):.4f}   "
          f"Macro-F1 {np.mean(f1_scores):.4f}+/-{np.std(f1_scores):.4f}   AUC {roc_macro:.4f}")

print("\n\n================ ITEM 1 SUMMARY ================")
print(f"{'Model':<26}{'Acc':>14}{'Macro-F1':>16}")
for tag, _, _ in configs:
    a_m, a_s, f_m, f_s, _ = summary[tag]
    print(f"{tag:<26}{a_m:.3f}+/-{a_s:.3f}   {f_m:.3f}+/-{f_s:.3f}")
print(f"{'Full ROCCO (reference)':<26}{'0.924':>10}      {'0.926':>10}")


##### BETO as-is  (freeze_lower6=True, epochs=3) #####


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on 

  Fold 1: Acc=0.918  F1=0.917


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on 

  Fold 7: Acc=0.959  F1=0.959


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on 

  Fold 8: Acc=0.902  F1=0.904
  -> Acc 0.9347+/-0.0182   Macro-F1 0.9361+/-0.0176   AUC 0.9879


================ ITEM 1 SUMMARY ================
Model                                Acc        Macro-F1
BETO as-is                0.912+/-0.021   0.914+/-0.021
Better-tuned BETO         0.935+/-0.018   0.936+/-0.018
Full ROCCO (reference)         0.924           0.926


## Cross-domain evaluation (out-of-domain benchmark)

Same protocol ROCCO uses for its benchmark: train BETO on the **full augmented `df`** (the in-domain experiment above was 8-fold CV; this trains on everything), then evaluate on the **leakage-filtered FakeNewsCorpusSpanish** test set. This is the generalization test — how well a text-only model transfers to an external corpus it never saw. Replace `<fill in>` in the summary with ROCCO's own benchmark numbers for the side-by-side.

In [9]:
# --- Build the cross-domain benchmark exactly as ROCCO does ---
# Clone FakeNewsCorpusSpanish, assemble df_test, then drop any article that
# overlaps the training set (by URL or title) to prevent leakage.
!git clone https://github.com/jpposadas/FakeNewsCorpusSpanish.git
import pandas as pd, numpy as np, re

columns = ["CATEGORY", "SOURCE", "HEADLINE", "TEXT", "LINK"]
test = pd.read_excel("/content/FakeNewsCorpusSpanish/test.xlsx", usecols=columns)

df_test = pd.DataFrame()
df_test["TITULO"] = test["HEADLINE"].fillna("").astype(str)
df_test["CORPUS"] = (test["HEADLINE"].fillna("").astype(str) + ". " + test["TEXT"].fillna("").astype(str))
df_test["AUTOR"] = test["SOURCE"].fillna("desconocido").astype(str)
df_test["URL"] = test["LINK"].fillna("https://unknown.com").astype(str)
df_test["FECHA"] = "2025-01-01"  # dummy if missing
df_test["LABEL"] = (
    test["CATEGORY"].astype(str).str.lower()
    .map({"true": 0, "real": 0, "false": 1, "fake": 1, "satira": 2})
    .astype(int)
)

def normalize_url(u):
    u = str(u).lower().strip().rstrip("/")
    u = re.sub(r'^https?://', '', u)
    return u
def normalize_title(t):
    return re.sub(r'\s+', ' ', str(t).lower().strip())

train_urls   = set(df["URL"].map(normalize_url))
train_titles = set(df["TITULO"].map(normalize_title))
leaked = df_test["URL"].map(normalize_url).isin(train_urls) | df_test["TITULO"].map(normalize_title).isin(train_titles)
print(f"Removed {leaked.sum()} overlapping articles ({leaked.sum()/len(df_test):.1%} of benchmark test set)")
df_test = df_test[~leaked].reset_index(drop=True)
df_test.head()

fatal: destination path 'FakeNewsCorpusSpanish' already exists and is not an empty directory.
Removed 243 overlapping articles (42.5% of benchmark test set)


,TITULO,CORPUS,AUTOR,URL,FECHA,LABEL
0,El Gobierno podrá acceder a las IPs de los móv...,El Gobierno podrá acceder a las IPs de los móv...,El matinal,https://www.elmatinal.com/espana-ultima-hora/e...,2025-01-01,1
1,,. Se han dado a conocer los datos electorales ...,AFPFactual,https://perma.cc/GYE6-SPMB,2025-01-01,1
2,,. Boooomm\nMUJERES VACUNADAS DE COVID ESTÁN MO...,AFPFactual,https://www.facebook.com/901924190177223/posts...,2025-01-01,1
3,Franja naranja de estado de guerra en chile,Franja naranja de estado de guerra en chile. L...,AFPFactual,"Perma | Presidente de Chile, hace el ridículo ...",2025-01-01,1
4,Es falso que la vacuna del coronavirus esteril...,Es falso que la vacuna del coronavirus esteril...,El Universo,https://www.eluniverso.com/larevista/2020/12/2...,2025-01-01,0


In [10]:
# =====================================================================
# Cross-domain (out-of-domain) BETO-only evaluation
# Mirrors ROCCO's benchmark protocol: train on the FULL augmented df
# (same recipe as the final retrain), evaluate on the leakage-filtered
# FakeNewsCorpusSpanish test set. Same two configs, no graph.
# =====================================================================
from sklearn.metrics import accuracy_score, f1_score, classification_report

configs = [
    ("BETO as-is",        True,  3),   # (tag, freeze_lower6, epochs)
    ("Better-tuned BETO", False, 6),
]

label_names = {0: "true", 1: "false", 2: "satira"}
y_true_xd = df_test["LABEL"].values
present = sorted(np.unique(y_true_xd))                 # benchmark is typically binary -> [0, 1]
present_names = [label_names[i] for i in present]
print(f"Benchmark classes present: {present_names}   n = {len(y_true_xd)}")

xd_summary = {}
for tag, freeze_lower6, epochs in configs:
    print(f"\n##### CROSS-DOMAIN: {tag}  (freeze_lower6={freeze_lower6}, epochs={epochs}) #####")

    enc_tr = tokenizer(list(df['CORPUS']), truncation=True, padding=True,
                       max_length=256, return_tensors="pt").to(device)
    enc_xd = tokenizer(list(df_test['CORPUS']), truncation=True, padding=True,
                       max_length=256, return_tensors="pt").to(device)
    ytr = torch.tensor(df['label'].values, dtype=torch.long).to(device)

    beto = AutoModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased").to(device)
    if freeze_lower6:
        for name, param in beto.named_parameters():
            if any(f"encoder.layer.{i}" in name for i in range(6)):
                param.requires_grad = False

    cls_head = nn.Linear(768, 3).to(device)
    params = [p for p in beto.parameters() if p.requires_grad] + list(cls_head.parameters())
    opt = AdamW(params, lr=2e-5)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    train_loader = DataLoader(
        TensorDataset(enc_tr['input_ids'], enc_tr['attention_mask'], ytr),
        batch_size=8, shuffle=True)
    amp_scaler = torch.cuda.amp.GradScaler()

    # === TRAIN on the full augmented df ===
    beto.train(); cls_head.train()
    for epoch in range(epochs):
        for input_ids, attn, labels in train_loader:
            opt.zero_grad()
            with torch.cuda.amp.autocast():
                out = beto(input_ids=input_ids, attention_mask=attn)
                loss = crit(cls_head(out.last_hidden_state[:, 0, :]), labels)
            amp_scaler.scale(loss).backward()
            amp_scaler.step(opt); amp_scaler.update()

    # === EVALUATE on the benchmark (no graph) ===
    beto.eval(); cls_head.eval()
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            logits_xd = cls_head(beto(**enc_xd).last_hidden_state[:, 0, :]).float()
        pred = logits_xd.argmax(dim=1).cpu().numpy()

    acc = accuracy_score(y_true_xd, pred)
    f1 = f1_score(y_true_xd, pred, labels=present, average='macro')
    xd_summary[tag] = (acc, f1)
    print(f"  Acc={acc:.3f}  Macro-F1={f1:.3f}")
    print(classification_report(y_true_xd, pred, labels=present,
                                target_names=present_names, zero_division=0))

    del beto, cls_head, enc_tr, enc_xd, train_loader
    torch.cuda.empty_cache(); gc.collect()

print("\n\n======== CROSS-DOMAIN SUMMARY (FakeNewsCorpusSpanish) ========")
print(f"{'Model':<26}{'Acc':>12}{'Macro-F1':>14}")
for tag, _, _ in configs:
    a, f = xd_summary[tag]
    print(f"{tag:<26}{a:.3f}{'':>7}{f:.3f}")
print(f"{'Full ROCCO (reference)':<26}{'<fill in>':>12}{'<fill in>':>14}")

Benchmark classes present: ['true', 'false']   n = 329

##### CROSS-DOMAIN: BETO as-is  (freeze_lower6=True, epochs=3) #####


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on 

  Acc=0.559  Macro-F1=0.598
              precision    recall  f1-score   support

        true       0.63      0.78      0.70       150
       false       0.74      0.37      0.50       179

   micro avg       0.67      0.56      0.61       329
   macro avg       0.69      0.58      0.60       329
weighted avg       0.69      0.56      0.59       329


##### CROSS-DOMAIN: Better-tuned BETO  (freeze_lower6=False, epochs=6) #####


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on 

  Acc=0.541  Macro-F1=0.586
              precision    recall  f1-score   support

        true       0.64      0.64      0.64       150
       false       0.64      0.46      0.53       179

   micro avg       0.64      0.54      0.59       329
   macro avg       0.64      0.55      0.59       329
weighted avg       0.64      0.54      0.58       329



======== CROSS-DOMAIN SUMMARY (FakeNewsCorpusSpanish) ========
Model                              Acc      Macro-F1
BETO as-is                0.559       0.598
Better-tuned BETO         0.541       0.586
Full ROCCO (reference)       <fill in>     <fill in>
